# Módulo 11.2 — Graph Autoencoders
**Sergio Gevatschnaider** · red de seis actores ficticios, no relaciones reales.

Objetivo: entrenar un Graph Autoencoder sencillo, reconstruir enlaces y evitar fuga de aristas de prueba. El encoder usa convolución de grafo y el decoder producto interno.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
torch.manual_seed(21);np.random.seed(21)
names=['IA A','IA B','IA C','Nube','Chips','Centro I+D']
features=torch.tensor([[1,.2],[.87,.35],[.75,.46],[.28,.9],[.37,.94],[.62,.7]],dtype=torch.float32)
positive=[(0,3),(0,4),(1,3),(1,4),(2,4),(2,5),(3,5)]
heldout=[(0,4),(1,4)]
train_pos=[e for e in positive if e not in heldout]
negatives=[(0,1),(0,2),(0,5),(1,2),(1,5),(2,3),(3,4),(4,5)]
train_neg=negatives[:6]; test_neg=negatives[6:]
def make_A(edges):
    A=torch.zeros((6,6))
    for i,j in edges: A[i,j]=A[j,i]=1
    return A
Atrain=make_A(train_pos)
I=torch.eye(6);Atilde=Atrain+I
D=torch.diag(torch.pow(Atilde.sum(1),-.5))
Anorm=D@Atilde@D
print('Aristas de entrenamiento',train_pos,'aristas de prueba',heldout)

## Entrenar el encoder con la adyacencia de entrenamiento, no la matriz completa
Las aristas de test no participan en la agregación. Se utiliza BCEWithLogitsLoss con pares observados de entrenamiento.

In [ ]:
class SmallGAE(nn.Module):
    def __init__(self):
        super().__init__(); self.W1=nn.Linear(2,8,bias=False); self.W2=nn.Linear(8,2,bias=False);self.bias=nn.Parameter(torch.tensor(-1.))
    def embed(self):return self.W2(torch.relu(Anorm@self.W1(features)))
    def logits(self,pairs):
        z=self.embed();return torch.stack([(z[i]*z[j]).sum()+self.bias for i,j in pairs])
model=SmallGAE();optimizer=torch.optim.Adam(model.parameters(),lr=.025,weight_decay=.001)
pairs=train_pos+train_neg
y=torch.tensor([1.]*len(train_pos)+[0.]*len(train_neg))
loss_fn=nn.BCEWithLogitsLoss();history=[]
for epoch in range(350):
    model.train();optimizer.zero_grad();loss=loss_fn(model.logits(pairs),y);loss.backward();optimizer.step();history.append(loss.item())
plt.plot(history);plt.xlabel('Época');plt.ylabel('BCE train');plt.show()

## Evaluación retenida y lectura crítica
El conjunto es minúsculo: los resultados no justifican conclusiones estadísticas ni inferencias sobre actores reales.

In [ ]:
model.eval()
with torch.no_grad():
    test_pairs=heldout+test_neg
    probs=torch.sigmoid(model.logits(test_pairs)).numpy()
print([(names[i],names[j],round(float(p),3)) for (i,j),p in zip(test_pairs,probs)])
from sklearn.metrics import roc_auc_score,average_precision_score
truth=np.array([1]*len(heldout)+[0]*len(test_neg))
print('AUC (muy inestable, n pequeño):',round(roc_auc_score(truth,probs),3))
print('AP (prevalencia artificial):',round(average_precision_score(truth,probs),3))

## Ejemplo conceptual de dos capas de relaciones
Cooperación y competencia deben codificarse mediante matrices separadas y requieren decoders por relación; el modelo anterior estima una sola capa.

In [ ]:
competition=[(0,1),(0,2),(1,2),(3,4)]
Acoord=make_A(positive);Acomp=make_A(competition)
print('Coordinación:',int(Acoord.sum().item()/2),'Competencia:',int(Acomp.sum().item()/2))
print('Son capas distintas:',not torch.equal(Acoord,Acomp))

## Preguntas de investigación
1. ¿Cómo definirías empíricamente cada relación con fuentes fechadas?
2. ¿Cómo controlarías la fuga de enlaces de test al encoder?
3. ¿Cuándo elegirías un decoder bilineal asimétrico?
4. ¿Qué ecuaciones dinámicas y observaciones necesitarías para contrastar Reina Roja?

## Experimento ampliado: evaluación de enlaces en un grafo sintético de 90 nodos
El ejemplo introductorio de seis actores tiene solo dos aristas de test y métricas muy inestables. Para un protocolo más serio construimos una red estocástica por bloques de 90 nodos con relaciones **ficticias**. Separamos pares positivos en entrenamiento, validación y test y generamos negativos **disjuntos**. El encoder solo observa la adyacencia de train. El objetivo sigue siendo didáctico: una sola red sintética no es inferencia estadística sobre vínculos reales.

In [ ]:
import copy
from sklearn.metrics import roc_auc_score,average_precision_score
rng=np.random.default_rng(2026);torch.manual_seed(2026)
N=90;blocks=np.repeat([0,1],45)
all_pairs=np.array([(i,j) for i in range(N) for j in range(i+1,N)],dtype=np.int64)
mask=rng.random(len(all_pairs))<np.array([.115 if blocks[i]==blocks[j] else .012 for i,j in all_pairs])
pos=all_pairs[mask]; neg=all_pairs[~mask]
rng.shuffle(pos);rng.shuffle(neg)
nval=max(1,int(.15*len(pos)));ntest=max(1,int(.15*len(pos)))
epos_val=pos[:nval]; epos_test=pos[nval:nval+ntest];epos_train=pos[nval+ntest:]
eneg_train=neg[:len(epos_train)]
eneg_val=neg[len(epos_train):len(epos_train)+len(epos_val)]
eneg_test=neg[len(epos_train)+len(epos_val):len(epos_train)+len(epos_val)+len(epos_test)]
assert len(pos)>=50 and len(neg)>=len(pos)
assert len({tuple(e) for group in [epos_train,epos_val,epos_test,eneg_train,eneg_val,eneg_test] for e in group})==len(pos)*2
A90=torch.zeros(N,N)
for i,j in epos_train:A90[i,j]=1;A90[j,i]=1
for i,j in np.concatenate([epos_val,epos_test]):assert A90[i,j]==0
degree=(A90+torch.eye(N)).sum(1)
d=torch.diag(degree.pow(-.5));N90=d@(A90+torch.eye(N))@d
X90=torch.tensor(np.c_[blocks==0,blocks==1,rng.normal(0,.5,N)],dtype=torch.float32)
print('Nodos:',N,'positivos:',len(pos),'train / val / test:',*[len(q) for q in [epos_train,epos_val,epos_test]])
print('Negativos por split: misma cantidad que positivos; no inferir probabilidades poblacionales de la calibración de scores.')

### Graph convolutional encoder y decoder de producto interno
Implementación mínima sin PyG: \(H=\operatorname{ReLU}(\widetilde D^{-1/2}\widetilde A\widetilde D^{-1/2}XW_0)\), \(Z=\widetilde D^{-1/2}\widetilde A\widetilde D^{-1/2}HW_1\). 
El decoder de producto interno es simétrico: \(\hat A_{ij}=\sigma(z_i^\top z_j+b)\).
Las aristas positivas de validación/test **no** entran en \(\widetilde A\).

In [ ]:
class ProperGAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.w0=nn.Linear(3,16,bias=False)
        self.w1=nn.Linear(16,8,bias=False)
        self.offset=nn.Parameter(torch.tensor(-1.))
    def embed(self):
        h=torch.relu(N90@self.w0(X90))
        return N90@self.w1(h)
    def logits(self,pairs):
        z=self.embed()
        p=torch.as_tensor(pairs,dtype=torch.long)
        return (z[p[:,0]]*z[p[:,1]]).sum(1)+self.offset
def pairset(pos,neg):
    pairs=np.concatenate([pos,neg])
    label=np.r_[np.ones(len(pos)),np.zeros(len(neg))].astype('float32')
    return pairs,torch.tensor(label)
train_pairs,train_y=pairset(epos_train,eneg_train)
valid_pairs,valid_y=pairset(epos_val,eneg_val)
test_pairs,test_y=pairset(epos_test,eneg_test)
deep=ProperGAE()
optim=torch.optim.Adam(deep.parameters(),lr=.015,weight_decay=.0005)
criterion=nn.BCEWithLogitsLoss()
best_ap=-1;best_state=None;stall=0;hist=[]
for epoch in range(450):
    deep.train();optim.zero_grad()
    loss=criterion(deep.logits(train_pairs),train_y)
    loss.backward();optim.step()
    deep.eval()
    with torch.no_grad():valprob=torch.sigmoid(deep.logits(valid_pairs)).numpy()
    val_ap=average_precision_score(valid_y.numpy(),valprob)
    hist.append([loss.item(),val_ap])
    if val_ap>best_ap+1e-4:
        best_ap=val_ap;best_state=copy.deepcopy(deep.state_dict());stall=0
    else:stall+=1
    if stall>=75:break
deep.load_state_dict(best_state)
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].plot(np.array(hist)[:,0]);axes[0].set(title='BCE train',xlabel='Épocas')
axes[1].plot(np.array(hist)[:,1]);axes[1].set(title='AP validación',xlabel='Épocas')
plt.tight_layout();plt.show()
print('Mejor AP validación:',round(best_ap,3),'épocas:',len(hist))

### Evaluación retenida y línea base topológica
El test se abre únicamente después de seleccionar el modelo. **Common Neighbors** (vecinos comunes) usa la misma adyacencia de entrenamiento y constituye una línea base sencilla. Reportamos AUROC y AP sobre pares balanceados artificialmente: estos valores NO se extrapolan a la prevalencia real de enlaces.

In [ ]:
deep.eval()
with torch.no_grad():test_prob=torch.sigmoid(deep.logits(test_pairs)).numpy()
y_true=test_y.numpy().astype(int)
A_np=A90.numpy()
cn=np.array([(A_np[i]*A_np[j]).sum() for i,j in test_pairs])
print('GAE   test ROC-AUC:',round(roc_auc_score(y_true,test_prob),3),
      'AP:',round(average_precision_score(y_true,test_prob),3))
print('CN    test ROC-AUC:',round(roc_auc_score(y_true,cn),3),
      'AP:',round(average_precision_score(y_true,cn),3))
print('La prevalencia en test es artificialmente 50%; AP fuera de este muestreo puede variar.')

## Grafo multinivel: decoder relacional y dirección
La capa «coordinación» y la capa «competencia» son conceptos diferentes: para relaciones dirigidas, un decoder bilineal \(z_i^\top R_r z_j\) puede producir valores distintos cuando intercambiamos i y j. **No estamos estimando vínculos reales**. Para un sistema empírico se necesitan aristas verificadas, tiempos y fuente.

In [ ]:
z_i=np.array([1.,.3]);z_j=np.array([.2,1.])
R_coop=np.array([[1.,0.],[0.,1.]])
R_direct=np.array([[0.,1.],[.1,0.]])
sigmoid=lambda x:1/(1+np.exp(-x))
for name,R in [('producto interno (simétrico)',R_coop),('relacional (asimétrico)',R_direct)]:
    p_ij=sigmoid(z_i@R@z_j);p_ji=sigmoid(z_j@R@z_i)
    print(name,'p(i→j)=',round(p_ij,4),'p(j→i)=',round(p_ji,4))
assert abs(sigmoid(z_i@R_coop@z_j)-sigmoid(z_j@R_coop@z_i))<1e-12
assert abs(sigmoid(z_i@R_direct@z_j)-sigmoid(z_j@R_direct@z_i))>1e-5

## Reina Roja: hipótesis dinámica independiente del GAE
Un autoencoder de grafos **estático** no prueba coevolución competitiva. El siguiente sistema temporal se introduce solo como ecuación hipotética de respuesta competitiva:
\(c_i(t+1)=c_i(t)+\alpha_i+\beta_i\max(0,c_j(t)-c_i(t))\).
La hipótesis exigiría datos longitudinales y una estrategia de identificación para contrastarse en el mundo real.

In [ ]:
a,b=1.0,.6
t_path=[]
for t in range(25):
    t_path.append((t,a,b))
    a_next=a+.035+.12*max(0,b-a)
    b_next=b+.045+.12*max(0,a-b)
    a,b=a_next,b_next
t_path=np.array(t_path)
plt.plot(t_path[:,0],t_path[:,1],label='Actor A sintético')
plt.plot(t_path[:,0],t_path[:,2],label='Actor B sintético')
plt.xlabel('Período');plt.ylabel('Capacidad ficticia');plt.title('Ejemplo de dinámica, no una estimación GAE')
plt.legend();plt.show()

## Checklist para publicación académica
1. Construir un diccionario de aristas con actor A, actor B, tipo, fecha, dirección, fuente y confianza.
2. Definir si el objetivo es recuperar enlaces ocultos del mismo período o anticipar nuevos enlaces futuros.
3. Para anticipación real, dividir por tiempo; ocultar cada enlace test del grafo usado para enviar mensajes cuando corresponda.
4. Mantener negativos disjuntos y reportar cómo fueron muestreados; considerar enlaces falsamente ausentes.
5. Comparar GAE con common neighbors, Adamic–Adar, atributos sin grafo y reglas simples.
6. Reportar ROC-AUC/AP, precisión@k, incertidumbre por semillas, sesgo de cobertura y prevalencia.
7. Reservar el estudio Reina Roja para series y datos adicionales: embeddings estáticos no identifican dinámica.
8. No interpretar vínculos sintéticos como contratos o inversiones efectivas.

**Fuentes primarias:** Kipf & Welling (2016) https://arxiv.org/abs/1611.07308 · GCN https://arxiv.org/abs/1609.02907 · CS224W Stanford https://web.stanford.edu/class/cs224w/.
